# tensorcas — Dedup Analysis

Measures tensor-level deduplication across three frameworks:
- **No-op fast path rate**: what fraction of tensor-steps are byte-identical to the previous checkpoint
- **Chunk-level reuse**: for changed tensors, what fraction of chunks already exist in the CAS
- **Cross-run sharing**: chunk overlap between two independent training runs

In [1]:
!pip install -q git+https://github.com/Olamyy/tensorcas.git@hash-cache-no-op-path zstandard xgboost scikit-learn torch

In [2]:
import copy
import json
import os
import tempfile
import time
from collections import defaultdict
from typing import Dict, List, Set

import numpy as np
import zstandard as zstd

from tensorcas.hashing import hash_chunk as _hash
from tensorcas.chunking import chunk_bytes as _chunk
from tensorcas.serialization import tensor_to_bytes as _tensor_to_bytes

def _to_bytes(arr: np.ndarray) -> bytes:
    raw, _, _ = _tensor_to_bytes(arr)
    return raw

_cctx = zstd.ZstdCompressor(level=3)
CHUNK_SIZE = 256 * 1024  # 256 KB

print("Imports OK")

Imports OK


## Tensor extraction helpers

These mirror the production adapters but operate directly on model objects rather than going through `tensorcasStore`.

In [3]:
def _extract_sklearn(model) -> Dict[str, np.ndarray]:
    tensors = {}
    for i, col in enumerate(model.estimators_):
        for j, tree in enumerate(col):
            idx = i * len(col) + j
            t = tree.tree_
            tensors[f"tree_{idx:06d}_features"] = t.feature.astype(np.int32)
            tensors[f"tree_{idx:06d}_thresholds"] = t.threshold.astype(np.float64)
            tensors[f"tree_{idx:06d}_values"] = t.value.squeeze().astype(np.float64)
    return dict(sorted(tensors.items()))


def _extract_xgboost(booster) -> Dict[str, np.ndarray]:
    with tempfile.NamedTemporaryFile(suffix=".json", delete=False) as f:
        tmp = f.name
    try:
        booster.save_model(tmp)
        with open(tmp) as f:
            model_json = json.load(f)
    finally:
        os.unlink(tmp)
    trees = model_json["learner"]["gradient_booster"]["model"].pop("trees")
    skeleton_bytes = json.dumps(model_json).encode()
    tensors = {"__skeleton__": np.frombuffer(skeleton_bytes, dtype=np.uint8).copy()}
    for i, tree in enumerate(trees):
        tb = json.dumps(tree).encode()
        tensors[f"tree_{i:06d}"] = np.frombuffer(tb, dtype=np.uint8).copy()
    return dict(sorted(tensors.items()))


def _extract_pytorch(state_dict) -> Dict[str, np.ndarray]:
    tensors = {}
    for k, v in state_dict.items():
        arr = v.detach().cpu().numpy()
        tensors[k] = np.ascontiguousarray(arr)
    return dict(sorted(tensors.items()))

print("Extraction helpers OK")

Extraction helpers OK


## Training helpers

Each function returns a list of model snapshots (one per step). Run 2 variants use a different random seed to simulate independent experiments sharing the same store.

In [4]:
def _train_sklearn_sequence(n_steps: int = 10, trees_per_step: int = 10):
    from sklearn.ensemble import GradientBoostingClassifier
    rng = np.random.default_rng(0)
    X = rng.standard_normal((500, 10)).astype(np.float32)
    y = (X[:, 0] > 0).astype(int)
    model = GradientBoostingClassifier(n_estimators=trees_per_step, warm_start=True, random_state=0)
    models = []
    for step in range(n_steps):
        model.set_params(n_estimators=(step + 1) * trees_per_step)
        model.fit(X, y)
        models.append(copy.deepcopy(model))
    return models


def _train_sklearn_run2(n_steps: int = 10, trees_per_step: int = 10):
    from sklearn.ensemble import GradientBoostingClassifier
    rng = np.random.default_rng(99)
    X = rng.standard_normal((500, 10)).astype(np.float32)
    y = (X[:, 0] > 0).astype(int)
    model = GradientBoostingClassifier(n_estimators=trees_per_step, warm_start=True, random_state=99)
    models = []
    for step in range(n_steps):
        model.set_params(n_estimators=(step + 1) * trees_per_step)
        model.fit(X, y)
        models.append(copy.deepcopy(model))
    return models

print("sklearn training helpers OK")

sklearn training helpers OK


In [5]:
def _train_xgboost_sequence(n_steps: int = 10, rounds_per_step: int = 10):
    import xgboost as xgb
    rng = np.random.default_rng(0)
    X = rng.standard_normal((500, 10)).astype(np.float32)
    y = (X[:, 0] > 0).astype(np.float32)
    dtrain = xgb.DMatrix(X, label=y)
    params = {"max_depth": 3, "objective": "binary:logistic", "seed": 0, "verbosity": 0}
    boosters, booster = [], None
    for _ in range(n_steps):
        booster = xgb.train(params, dtrain, num_boost_round=rounds_per_step, xgb_model=booster, verbose_eval=False)
        boosters.append(booster)
    return boosters


def _train_xgboost_run2(n_steps: int = 10, rounds_per_step: int = 10):
    import xgboost as xgb
    rng = np.random.default_rng(99)
    X = rng.standard_normal((500, 10)).astype(np.float32)
    y = (X[:, 0] > 0).astype(np.float32)
    dtrain = xgb.DMatrix(X, label=y)
    params = {"max_depth": 3, "objective": "binary:logistic", "seed": 99, "verbosity": 0}
    boosters, booster = [], None
    for _ in range(n_steps):
        booster = xgb.train(params, dtrain, num_boost_round=rounds_per_step, xgb_model=booster, verbose_eval=False)
        boosters.append(booster)
    return boosters

print("XGBoost training helpers OK")

XGBoost training helpers OK


In [6]:
def _make_pt_model(device):
    import torch.nn as nn
    return nn.Sequential(
        nn.Linear(64, 256), nn.ReLU(),
        nn.Linear(256, 256), nn.ReLU(),
        nn.Linear(256, 10),
    ).to(device)


def _train_pytorch_sequence(device, n_epochs: int = 20, freeze_epoch: int = 10):
    import torch
    import torch.nn as nn
    rng = np.random.default_rng(42)
    X = torch.from_numpy(rng.standard_normal((1000, 64)).astype(np.float32)).to(device)
    y = torch.from_numpy(rng.integers(0, 10, 1000).astype(np.int64)).to(device)
    model = _make_pt_model(device)
    opt = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    crit = nn.CrossEntropyLoss()
    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X, y),
        batch_size=64, shuffle=True,
        generator=torch.Generator().manual_seed(0), num_workers=0,
    )
    frozen = False
    state_dicts = []
    model.train()
    for epoch in range(1, n_epochs + 1):
        if epoch == freeze_epoch + 1 and not frozen:
            for layer in [model[0], model[2]]:
                for p in layer.parameters():
                    p.requires_grad_(False)
            opt = torch.optim.SGD([p for p in model.parameters() if p.requires_grad], lr=0.001, momentum=0.9)
            frozen = True
        for xb, yb in loader:
            opt.zero_grad()
            crit(model(xb), yb).backward()
            opt.step()
        state_dicts.append({k: v.clone().cpu() for k, v in model.state_dict().items()})
    return state_dicts


def _train_pytorch_base(device, n_epochs: int = 5):
    import torch
    import torch.nn as nn
    rng = np.random.default_rng(0)
    X = torch.from_numpy(rng.standard_normal((1000, 64)).astype(np.float32)).to(device)
    y = torch.from_numpy(rng.integers(0, 10, 1000).astype(np.int64)).to(device)
    model = _make_pt_model(device)
    opt = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    crit = nn.CrossEntropyLoss()
    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X, y),
        batch_size=64, shuffle=True,
        generator=torch.Generator().manual_seed(0), num_workers=0,
    )
    model.train()
    for _ in range(n_epochs):
        for xb, yb in loader:
            opt.zero_grad()
            crit(model(xb), yb).backward()
            opt.step()
    return {k: v.clone() for k, v in model.state_dict().items()}


def _train_pytorch_finetune(base_state, seed: int, device, n_epochs: int = 5):
    """Fine-tune from base_state. Returns only the fine-tuned checkpoints (not base)."""
    import torch
    import torch.nn as nn
    rng = np.random.default_rng(seed)
    X = torch.from_numpy(rng.standard_normal((1000, 64)).astype(np.float32)).to(device)
    y = torch.from_numpy(rng.integers(0, 10, 1000).astype(np.int64)).to(device)
    model = _make_pt_model(device)
    model.load_state_dict(base_state)
    opt = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)
    crit = nn.CrossEntropyLoss()
    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X, y),
        batch_size=64, shuffle=True,
        generator=torch.Generator().manual_seed(seed), num_workers=0,
    )
    state_dicts = []
    model.train()
    for _ in range(n_epochs):
        for xb, yb in loader:
            opt.zero_grad()
            crit(model(xb), yb).backward()
            opt.step()
        state_dicts.append({k: v.clone().cpu() for k, v in model.state_dict().items()})
    return state_dicts

print("PyTorch training helpers OK")

PyTorch training helpers OK


## Measurement functions

In [7]:
def measure_noop(sequences: List[Dict[str, np.ndarray]]) -> Dict:
    """Fraction of tensor-steps that are byte-identical to the previous checkpoint."""
    tensor_names = sorted(sequences[0].keys())
    stats = {n: {"identical": 0, "changed": 0, "compressed_bytes": []} for n in tensor_names}
    for prev, curr in zip(sequences, sequences[1:]):
        for name in tensor_names:
            if name not in curr:
                continue
            raw_p, raw_c = _to_bytes(prev[name]), _to_bytes(curr[name])
            if _hash(raw_p) == _hash(raw_c):
                stats[name]["identical"] += 1
            else:
                stats[name]["changed"] += 1
                stats[name]["compressed_bytes"].append(len(_cctx.compress(raw_c)))
    for name in tensor_names:
        if not stats[name]["compressed_bytes"]:
            stats[name]["compressed_bytes"].append(len(_cctx.compress(_to_bytes(sequences[0][name]))))
    n_pairs = len(sequences) - 1
    result = {}
    for name, s in stats.items():
        result[name] = {
            "identical": s["identical"],
            "changed": s["changed"],
            "identical_pct": s["identical"] / n_pairs * 100,
            "avg_compressed_bytes": sum(s["compressed_bytes"]) / len(s["compressed_bytes"]),
        }
    all_id = sum(s["identical"] for s in stats.values())
    all_tot = n_pairs * len(tensor_names)
    result["__summary__"] = {"identical_pct": all_id / all_tot * 100 if all_tot else 0.0}
    return result

print("measure_noop OK")

measure_noop OK


In [8]:
def measure_chunk_dedup(sequences: List[Dict[str, np.ndarray]], chunk_size: int) -> Dict:
    """For changed tensors, fraction of chunks already in the cumulative CAS store."""
    tensor_names = sorted(sequences[0].keys())
    changed: Dict[str, Dict] = defaultdict(lambda: {"reused": 0, "total": 0})
    cumulative_hashes: Dict[str, Set[str]] = defaultdict(set)
    for prev, curr in zip(sequences, sequences[1:]):
        for name in tensor_names:
            if name not in curr:
                continue
            raw_p, raw_c = _to_bytes(prev[name]), _to_bytes(curr[name])
            for c in _chunk(raw_p, chunk_size):
                cumulative_hashes[name].add(_hash(c))
            if _hash(raw_p) == _hash(raw_c):
                continue
            for c in _chunk(raw_c, chunk_size):
                changed[name]["total"] += 1
                if _hash(c) in cumulative_hashes[name]:
                    changed[name]["reused"] += 1
    result = {}
    tot_r, tot_t = 0, 0
    for name, s in changed.items():
        result[name] = {
            "total": s["total"],
            "reused": s["reused"],
            "reuse_pct": s["reused"] / s["total"] * 100 if s["total"] else 0.0,
        }
        tot_r += s["reused"]
        tot_t += s["total"]
    result["__summary__"] = {
        "reuse_pct": tot_r / tot_t * 100 if tot_t else 0.0,
        "no_changed_tensors": len(changed) == 0,
    }
    return result

print("measure_chunk_dedup OK")

measure_chunk_dedup OK


In [9]:
def measure_crossrun(
    seqs_r1: List[Dict[str, np.ndarray]],
    seqs_r2: List[Dict[str, np.ndarray]],
    chunk_size: int,
) -> Dict:
    """Chunk overlap between two independent training runs."""
    def _unique_hashes(seqs) -> Set[str]:
        s: Set[str] = set()
        for tensors in seqs:
            for arr in tensors.values():
                for c in _chunk(_to_bytes(arr), chunk_size):
                    s.add(_hash(c))
        return s
    h1, h2 = _unique_hashes(seqs_r1), _unique_hashes(seqs_r2)
    shared = len(h1 & h2)
    return {
        "run1_unique": len(h1), "run2_unique": len(h2),
        "shared": shared, "new_in_run2": len(h2) - shared,
        "reuse_pct": shared / len(h2) * 100 if h2 else 0.0,
    }

print("measure_crossrun OK")

measure_crossrun OK


## Print helpers

In [10]:
def _fmt_bytes(n: float) -> str:
    size = float(n)
    for unit in ("B", "KB", "MB", "GB"):
        if size < 1024:
            return f"{size:.1f}{unit}"
        size /= 1024
    return f"{size:.1f}TB"


def print_noop(label: str, stats: Dict) -> None:
    summary = stats.get("__summary__", {})
    print(f"\n[{label}] No-op fast path — tensor identity across steps")
    print(f"  Overall: {summary['identical_pct']:.1f}% of tensor-steps byte-identical")
    print(f"  {'Tensor':<36} {'Identical':>10} {'Changed':>8} {'Identical%':>11} {'AvgCompressed':>14}")
    print(f"  {'-'*36} {'-'*10} {'-'*8} {'-'*11} {'-'*14}")
    for name, s in sorted((k, v) for k, v in stats.items() if k != "__summary__"):
        print(f"  {name:<36} {s['identical']:>10} {s['changed']:>8} "
              f"{s['identical_pct']:>10.1f}% {_fmt_bytes(s['avg_compressed_bytes']):>14}")


def print_chunk(label: str, stats: Dict, chunk_size: int) -> None:
    cs = f"{chunk_size // 1024}KB" if chunk_size < 1024**2 else f"{chunk_size // 1024**2}MB"
    summary = stats.get("__summary__", {})
    print(f"\n[{label}] Chunk-level reuse in changed tensors (chunk={cs})")
    if summary.get("no_changed_tensors"):
        print("  Overall: N/A — no changed tensors")
        return
    print(f"  Overall: {summary['reuse_pct']:.1f}% of chunks in changed tensors already in store")
    print(f"  {'Tensor':<36} {'Total':>8} {'Reused':>8} {'Reuse%':>8}")
    print(f"  {'-'*36} {'-'*8} {'-'*8} {'-'*8}")
    for name, s in sorted((k, v) for k, v in stats.items() if k != "__summary__"):
        print(f"  {name:<36} {s['total']:>8} {s['reused']:>8} {s['reuse_pct']:>7.1f}%")


def print_crossrun(label: str, stats: Dict) -> None:
    print(f"\n[{label}] Cross-run chunk sharing")
    print(f"  Run 1 unique chunks : {stats['run1_unique']:,}")
    print(f"  Run 2 unique chunks : {stats['run2_unique']:,}")
    print(f"  Shared (R1 ∩ R2)    : {stats['shared']:,}  ({stats['reuse_pct']:.1f}%)")
    print(f"  New in run 2        : {stats['new_in_run2']:,}")

print("Print helpers OK")

Print helpers OK


## sklearn — GradientBoostingClassifier warm-start

In [11]:
t0 = time.time()
print("Training sklearn run 1 (seed=0)...", end=" ", flush=True)
sklearn_models_r1 = _train_sklearn_sequence(n_steps=10, trees_per_step=10)
print(f"{time.time() - t0:.1f}s")

t0 = time.time()
print("Training sklearn run 2 (seed=99)...", end=" ", flush=True)
sklearn_models_r2 = _train_sklearn_run2(n_steps=10, trees_per_step=10)
print(f"{time.time() - t0:.1f}s")

Training sklearn run 1 (seed=0)... 19.7s
Training sklearn run 2 (seed=99)... 0.0s


In [12]:
sklearn_seqs_r1 = [_extract_sklearn(m) for m in sklearn_models_r1]
print_noop("sklearn", measure_noop(sklearn_seqs_r1))


[sklearn] No-op fast path — tensor identity across steps
  Overall: 100.0% of tensor-steps byte-identical
  Tensor                                Identical  Changed  Identical%  AvgCompressed
  ------------------------------------ ---------- -------- ----------- --------------
  tree_000000_features                          9        0      100.0%          21.0B
  tree_000000_thresholds                        9        0      100.0%          33.0B
  tree_000000_values                            9        0      100.0%          26.0B
  tree_000001_features                          9        0      100.0%          29.0B
  tree_000001_thresholds                        9        0      100.0%          39.0B
  tree_000001_values                            9        0      100.0%          49.0B
  tree_000002_features                          9        0      100.0%          21.0B
  tree_000002_thresholds                        9        0      100.0%          33.0B
  tree_000002_values             

In [13]:
print_chunk("sklearn", measure_chunk_dedup(sklearn_seqs_r1, CHUNK_SIZE), CHUNK_SIZE)


[sklearn] Chunk-level reuse in changed tensors (chunk=256KB)
  Overall: N/A — no changed tensors


In [14]:
sklearn_seqs_r2 = [_extract_sklearn(m) for m in sklearn_models_r2]
print_crossrun("sklearn (seed=0 vs seed=99)", measure_crossrun(sklearn_seqs_r1, sklearn_seqs_r2, CHUNK_SIZE))


[sklearn (seed=0 vs seed=99)] Cross-run chunk sharing
  Run 1 unique chunks : 113
  Run 2 unique chunks : 116
  Shared (R1 ∩ R2)    : 1  (0.9%)
  New in run 2        : 115


## XGBoost — warm-start boosting

In [15]:
t0 = time.time()
print("Training XGBoost run 1 (seed=0)...", end=" ", flush=True)
xgb_boosters_r1 = _train_xgboost_sequence(n_steps=10, rounds_per_step=10)
print(f"{time.time() - t0:.1f}s")

t0 = time.time()
print("Training XGBoost run 2 (seed=99)...", end=" ", flush=True)
xgb_boosters_r2 = _train_xgboost_run2(n_steps=10, rounds_per_step=10)
print(f"{time.time() - t0:.1f}s")

Training XGBoost run 1 (seed=0)... 0.6s
Training XGBoost run 2 (seed=99)... 0.1s


In [16]:
xgb_seqs_r1 = [_extract_xgboost(b) for b in xgb_boosters_r1]
print_noop("xgboost", measure_noop(xgb_seqs_r1))


[xgboost] No-op fast path — tensor identity across steps
  Overall: 90.9% of tensor-steps byte-identical
  Tensor                                Identical  Changed  Identical%  AvgCompressed
  ------------------------------------ ---------- -------- ----------- --------------
  __skeleton__                                  0        9        0.0%         463.1B
  tree_000000                                   9        0      100.0%         297.0B
  tree_000001                                   9        0      100.0%         311.0B
  tree_000002                                   9        0      100.0%         320.0B
  tree_000003                                   9        0      100.0%         318.0B
  tree_000004                                   9        0      100.0%         315.0B
  tree_000005                                   9        0      100.0%         319.0B
  tree_000006                                   9        0      100.0%         319.0B
  tree_000007                     

In [17]:
print_chunk("xgboost", measure_chunk_dedup(xgb_seqs_r1, CHUNK_SIZE), CHUNK_SIZE)


[xgboost] Chunk-level reuse in changed tensors (chunk=256KB)
  Overall: 0.0% of chunks in changed tensors already in store
  Tensor                                  Total   Reused   Reuse%
  ------------------------------------ -------- -------- --------
  __skeleton__                                9        0     0.0%


In [18]:
xgb_seqs_r2 = [_extract_xgboost(b) for b in xgb_boosters_r2]
print_crossrun("xgboost (seed=0 vs seed=99)", measure_crossrun(xgb_seqs_r1, xgb_seqs_r2, CHUNK_SIZE))


[xgboost (seed=0 vs seed=99)] Cross-run chunk sharing
  Run 1 unique chunks : 110
  Run 2 unique chunks : 110
  Shared (R1 ∩ R2)    : 0  (0.0%)
  New in run 2        : 110


## PyTorch — MLP fine-tuning

Layers 0 and 2 are frozen after epoch 10. Only the head (layer 4) trains in epochs 11–20.
Cross-run uses a shared pre-trained base with two different fine-tuning seeds.

In [19]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: cpu


In [ ]:
t0 = time.time()
print("Training PyTorch 20-epoch sequence (freeze after epoch 10)...", end=" ", flush=True)
pt_state_dicts = _train_pytorch_sequence(device, n_epochs=20, freeze_epoch=10)
print(f"{time.time() - t0:.1f}s")

Training PyTorch 20-epoch sequence (freeze after epoch 10)... 

In [ ]:
pt_seqs = [_extract_pytorch(sd) for sd in pt_state_dicts]
print_noop("pytorch", measure_noop(pt_seqs))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({"font.size": 11, "figure.dpi": 150})

from pathlib import Path
FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(exist_ok=True)

# --- Compute no-op rates from the sequences already in memory ---
sklearn_noop_pct = measure_noop(sklearn_seqs_r1)["__summary__"]["identical_pct"]
xgb_noop_pct    = measure_noop(xgb_seqs_r1)["__summary__"]["identical_pct"]
pt_noop_pct     = measure_noop(pt_seqs)["__summary__"]["identical_pct"]

frameworks   = ["sklearn\n(warm-start)", "XGBoost\n(warm-start)", "PyTorch\n(full fine-tune)"]
noop_rates   = [sklearn_noop_pct, xgb_noop_pct, pt_noop_pct]
colors       = ["#4C72B0", "#DD8452", "#C44E52"]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(frameworks, noop_rates, color=colors, width=0.5)
for bar, val in zip(bars, noop_rates):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f"{val:.1f}%", ha="center", va="bottom", fontsize=11)
ax.set_ylabel("No-op rate (% of tensor-steps byte-identical)")
ax.set_title("No-op rate by framework\n(% of tensor-steps that skip I/O entirely)")
ax.set_ylim(0, 120)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "noop_rate_by_framework.png")
plt.show()
print(f"sklearn: {sklearn_noop_pct:.1f}%  |  xgboost: {xgb_noop_pct:.1f}%  |  pytorch: {pt_noop_pct:.1f}%")
print("Saved figures/noop_rate_by_framework.png")

In [ ]:
print_chunk("pytorch", measure_chunk_dedup(pt_seqs, CHUNK_SIZE), CHUNK_SIZE)

In [ ]:
t0 = time.time()
print("Training shared base + 2 fine-tune runs...", end=" ", flush=True)
pt_base = _train_pytorch_base(device)
pt_sds_r1 = _train_pytorch_finetune(pt_base, seed=42, device=device, n_epochs=5)
pt_sds_r2 = _train_pytorch_finetune(pt_base, seed=99, device=device, n_epochs=5)
print(f"{time.time() - t0:.1f}s")

In [ ]:
pt_seqs_r1 = [_extract_pytorch(sd) for sd in pt_sds_r1]
pt_seqs_r2 = [_extract_pytorch(sd) for sd in pt_sds_r2]
print_crossrun(
    "pytorch (fine-tuned only, seed=42 vs seed=99)",
    measure_crossrun(pt_seqs_r1, pt_seqs_r2, CHUNK_SIZE),
)

## DVC vs tensorcas — storage comparison

Simulates what DVC would store vs what tensorcas actually stores on disk.

**DVC-simulated bytes**: each checkpoint file compressed with zstd level 3, stored once per unique content hash (file-level dedup). Identical files share one entry; changed files are stored in full.

**tensorcas bytes**: actual `.chunk` file sizes in the CAS `objects/` directory after saving all checkpoints through `tensorcasStore` (tensor-level dedup).

Both sides use zstd level 3. The only variable is dedup granularity.

In [ ]:
import shutil
import tempfile
from pathlib import Path

from tensorcas.store import tensorcasStore
from tensorcas.adapters.sklearn import SklearnAdapter
from tensorcas.adapters.xgboost import XGBoostAdapter
from tensorcas.adapters.pytorch import PyTorchAdapter


def _dvc_bytes(files: list[Path]) -> int:
    """Simulate DVC file-level dedup: zstd each file, count unique compressed blobs."""
    seen: dict[str, int] = {}  # content_hash -> compressed_size
    cctx = zstd.ZstdCompressor(level=3)
    for path in files:
        raw = path.read_bytes()
        compressed = cctx.compress(raw)
        h = _hash(compressed)
        if h not in seen:
            seen[h] = len(compressed)
    return sum(seen.values())


def _tensorcas_bytes(root: Path) -> int:
    """Sum of all .chunk file sizes under {root}/objects/."""
    return sum(p.stat().st_size for p in (root / "objects").rglob("*.chunk"))


def _run_comparison(label: str, models: list, adapter, checkpoint_dir: Path) -> dict:
    checkpoint_files = sorted(checkpoint_dir.glob("*"))
    dvc = _dvc_bytes(checkpoint_files)

    with tempfile.TemporaryDirectory() as tmp:
        root = Path(tmp)
        with tensorcasStore(root=root, run_id="bench", adapter=adapter) as store:
            for step, model in enumerate(models, 1):
                store.save(model, step=step)
        tensorcas = _tensorcas_bytes(root)

    raw_total = sum(p.stat().st_size for p in checkpoint_files)
    savings_vs_dvc = (dvc - tensorcas) / dvc * 100 if dvc else 0.0
    return {
        "label": label,
        "checkpoints": len(models),
        "raw_total": raw_total,
        "dvc_bytes": dvc,
        "tensorcas_bytes": tensorcas,
        "ratio": tensorcas / dvc if dvc else 0.0,
        "savings_vs_dvc": savings_vs_dvc,
    }


print("DVC comparison helpers OK")

### Generate checkpoint files

Serializes the already-trained models to disk so the DVC comparison can measure file sizes. Uses the same formats the adapters expect: `.pkl` for sklearn, `.ubj` for XGBoost, `.pt` for PyTorch.

In [ ]:
import pickle
import torch

CHECKPOINT_DIR = Path("/tmp/tensorcas_checkpoints")

# sklearn
sklearn_dir = CHECKPOINT_DIR / "sklearn"
sklearn_dir.mkdir(parents=True, exist_ok=True)
for step, model in enumerate(sklearn_models_r1, 1):
    with open(sklearn_dir / f"step_{step * 10:06d}.pkl", "wb") as f:
        pickle.dump(model, f)
print(f"sklearn: {len(sklearn_models_r1)} checkpoints written to {sklearn_dir}")

# XGBoost
xgboost_dir = CHECKPOINT_DIR / "xgboost"
xgboost_dir.mkdir(parents=True, exist_ok=True)
for step, booster in enumerate(xgb_boosters_r1, 1):
    booster.save_model(str(xgboost_dir / f"step_{step * 10:06d}.ubj"))
print(f"xgboost: {len(xgb_boosters_r1)} checkpoints written to {xgboost_dir}")

# PyTorch
pytorch_dir = CHECKPOINT_DIR / "pytorch"
pytorch_dir.mkdir(parents=True, exist_ok=True)
for epoch, sd in enumerate(pt_state_dicts, 1):
    torch.save(sd, pytorch_dir / f"epoch_{epoch:06d}.pt")
print(f"pytorch: {len(pt_state_dicts)} checkpoints written to {pytorch_dir}")

In [ ]:
results = []

# sklearn
print("Running sklearn comparison...", end=" ", flush=True)
t0 = time.time()
results.append(_run_comparison(
    "sklearn", sklearn_models_r1, SklearnAdapter(),
    CHECKPOINT_DIR / "sklearn",
))
print(f"{time.time() - t0:.1f}s")

# XGBoost
print("Running XGBoost comparison...", end=" ", flush=True)
t0 = time.time()
results.append(_run_comparison(
    "xgboost", xgb_boosters_r1, XGBoostAdapter(),
    CHECKPOINT_DIR / "xgboost",
))
print(f"{time.time() - t0:.1f}s")

# PyTorch
print("Running PyTorch comparison...", end=" ", flush=True)
t0 = time.time()
pt_models = []
for sd in pt_state_dicts:
    m = _make_pt_model("cpu")
    m.load_state_dict({k: v.clone() if isinstance(v, torch.Tensor) else torch.tensor(v) for k, v in sd.items()})
    pt_models.append(m)
results.append(_run_comparison(
    "pytorch", pt_models, PyTorchAdapter(),
    CHECKPOINT_DIR / "pytorch",
))
print(f"{time.time() - t0:.1f}s")

In [ ]:
print(f"\n{'Framework':<12} {'Checkpoints':>12} {'Raw total':>10} {'DVC bytes':>10} {'tensorcas bytes':>10} {'Ratio':>7} {'vs DVC':>8}")
print("=" * 75)
for r in results:
    print(
        f"{r['label']:<12} {r['checkpoints']:>12} "
        f"{_fmt_bytes(r['raw_total']):>10} {_fmt_bytes(r['dvc_bytes']):>10} "
        f"{_fmt_bytes(r['tensorcas_bytes']):>10} {r['ratio']:>7.3f} "
        f"{r['savings_vs_dvc']:>7.1f}%"
    )